# Prompt sensitivity

Compare the original behavioral prompt (0) with the broad paraphrase (2)
Use nine nonbinary outcomes across 224 profiles, giving 2,016 comparisons per model
Zero differences under both prompts count as agreement

Knowledge probes compare exact direction categories across all 32 relationships
Their prompt versions are 0 and 1

Run from the project root or analysis folder


In [ ]:
from pathlib import Path
import os
import sys
import hashlib

ROOT = next((candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
             if (candidate / "analysis" / "analysis_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the project root or its analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
from analysis.analysis_utils import DATA_DIR, RESULT_DIR, MODELS, normalize_knowledge

EXPECTED_PROFILES = 224
BEHAVIOR_VERSIONS = [0, 2]
NONBINARY_MEASURES = ["Overdose", "Public_prop", "Recept_sharing", "Partners", "Drug_in",
                     "Drug_out", "Partner_HCV_prop", "Injections", "Alt_route"]
RAW_HASHES_BEFORE = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                     for p in DATA_DIR.iterdir() if p.suffix in {".csv", ".jsonl"}}

def require_unique(frame, keys):
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f"Missing or duplicate identifiers: {keys}.")


No confidence intervals are calculated


## Behavioral housing-difference signs

Match the same profiles under both housing conditions and prompt versions


In [ ]:
data = pd.read_csv(DATA_DIR / "outputs.csv")
data = data.loc[data.Paraphrase.isin(BEHAVIOR_VERSIONS)].copy()
require_unique(data, ["Model", "Paraphrase", "Agent", "Homeless"])
expected_groups = {(model, version, housing) for model in MODELS
                   for version in BEHAVIOR_VERSIONS for housing in [0, 1]}
if set(data.groupby(["Model", "Paraphrase", "Homeless"]).groups) != expected_groups:
    raise ValueError("Expected all three models, original/broad prompts, and both housing conditions.")
cohort = set(data.Agent)
if len(cohort) != EXPECTED_PROFILES:
    raise ValueError("Expected 224 individual profiles.")
for key, group in data.groupby(["Model", "Paraphrase", "Homeless"]):
    if set(group.Agent) != cohort:
        raise ValueError(f"Mismatched profile cohort: {key}.")
for measure in NONBINARY_MEASURES:
    data[measure] = pd.to_numeric(data[measure], errors="raise")
    if np.isinf(data[measure]).any() or data[measure].dropna().lt(0).any():
        raise ValueError(f"Invalid numeric output: {measure}.")

for measure in ["Public_prop", "Recept_sharing", "Partner_HCV_prop"]:
    if data[measure].dropna().gt(1).any():
        raise ValueError(f"Out-of-range proportion: {measure}.")

semantic_rows = []
for model, group in data.groupby("Model"):
    for measure in NONBINARY_MEASURES:
        wide = group.pivot(index="Agent", columns=["Paraphrase", "Homeless"], values=measure)
        columns = pd.MultiIndex.from_tuples([(0,0), (0,1), (2,0), (2,1)])
        paired = wide.reindex(columns=columns).replace([np.inf,-np.inf],np.nan).dropna()
        if len(paired) != EXPECTED_PROFILES:
            raise ValueError(f"{model}/{measure}: expected 224 complete original/broad profile pairs.")
        original = paired[(0,1)]-paired[(0,0)]
        altered = paired[(2,1)]-paired[(2,0)]
        same = np.sign(original).eq(np.sign(altered))
        nonzero = original.ne(0) & altered.ne(0)
        semantic_rows.append(dict(model=model, measure=measure, N=len(paired),
            directional_consistency=same.mean(), both_zero=int((original.eq(0)&altered.eq(0)).sum()),
            N_both_nonzero=int(nonzero.sum()),
            nonzero_direction_consistency=same[nonzero].mean() if nonzero.any() else np.nan))
semantic_all = pd.DataFrame(semantic_rows)
model_consistency = (semantic_all.groupby("model").directional_consistency.mean().mul(100)
                     .reset_index(name="mean_profile_sign_agreement_pct"))
RESULT_DIR.mkdir(exist_ok=True)
semantic_all.to_csv(RESULT_DIR / "prompt_semantic.csv", index=False)
model_consistency.to_csv(RESULT_DIR / "prompt_summary.csv", index=False)
print(semantic_all[["model","measure","N","directional_consistency","both_zero",
                    "N_both_nonzero","nonzero_direction_consistency"]].to_string(index=False))
print("\nOriginal versus broad behavioral wording: sign agreement across nine nonbinary outcomes")
totals = semantic_all.groupby("model").apply(lambda group: pd.Series(dict(
    N_comparisons=int(group.N.sum()), N_same=int(round((group.N*group.directional_consistency).sum())),
    Agreement_pct=float(100*(group.N*group.directional_consistency).sum()/group.N.sum()),
    N_both_zero=int(group.both_zero.sum()))))
print(totals.to_string())
print("Zero differences under both wordings count as agreement. This is profile-level sign preservation, not literature agreement.")


## Knowledge-probe direction categories

Any category change counts as disagreement, including mixed or uncertain responses


In [7]:
knowledge_df = pd.read_csv(DATA_DIR / "knowledge_outputs.csv")
knowledge_df = normalize_knowledge(knowledge_df)
keys = ["Model", "exposure", "outcome"]
require_unique(knowledge_df, keys + ["Paraphrase"])
knowledge_df["direction"] = knowledge_df["direction"].str.strip().str.lower()
if not knowledge_df["direction"].isin(["increase", "decrease", "mixed", "uncertain"]).all():
    raise ValueError("Invalid/missing knowledge direction; inspect original responses.")
orig = knowledge_df.query("Paraphrase == 0")[keys + ["direction"]].rename(columns={"direction":"direction_orig"})
para = knowledge_df.query("Paraphrase == 1")[keys + ["direction"]].rename(columns={"direction":"direction_para"})
merged = orig.merge(para, on=keys, how="outer", validate="one_to_one", indicator=True)
if not merged["_merge"].eq("both").all():
    raise ValueError("Original and paraphrase knowledge pairs do not match.")
merged["agreement"] = merged.direction_orig.eq(merged.direction_para)
merged["opposite_direction"] = ((merged.direction_orig.eq("increase") & merged.direction_para.eq("decrease")) |
                               (merged.direction_orig.eq("decrease") & merged.direction_para.eq("increase")))
agreement = merged.groupby("Model")["agreement"].agg(agreement="sum", total="count").reset_index()
agreement["agreement_pct"] = 100 * agreement.agreement / agreement.total
print("Exact categorical agreement (mixed/uncertain changes are disagreements):")
print(agreement.to_string(index=False))
print(merged.loc[~merged.agreement, keys + ["direction_orig", "direction_para"]].to_string(index=False))
assert agreement.total.eq(32).all()
agreement.to_csv(RESULT_DIR / "knowledge_prompt_agreement.csv", index=False)
merged.loc[~merged.agreement].to_csv(RESULT_DIR / "knowledge_prompt_discrepancies.csv", index=False)



Exact categorical agreement (mixed/uncertain changes are disagreements):
          Model  agreement  total  agreement_pct
claudesonnet4.6         31     32         96.875
  deepseek.v3.2         31     32         96.875
   gpt-oss-120b         28     32           87.5
          Model                                                         exposure                                                          outcome direction_orig direction_para
claudesonnet4.6                                   current or recent homelessness non-injection routes of opioid administration (smoking/snorting)       decrease      uncertain
  deepseek.v3.2 non-injection routes of opioid administration (smoking/snorting)                     personal HCV infection status (RNA-positive)       decrease       increase
   gpt-oss-120b                        engaging in opioid use disorder treatment non-injection routes of opioid administration (smoking/snorting)          mixed       increase
   gpt-oss-120b non-injecti

## Save results and check raw files

Save four comparison CSVs for the paper tables
Verify that raw responses are unchanged


In [9]:
RAW_HASHES_AFTER = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in DATA_DIR.iterdir() if p.suffix in {".csv", ".jsonl"}}
assert RAW_HASHES_AFTER == RAW_HASHES_BEFORE, "Raw saved-response files changed."
print("Saved four prompt-sensitivity CSVs. All raw CSV/JSONL files are unchanged.")


Saved four prompt-sensitivity CSVs. All raw CSV/JSONL files are unchanged.
